# About this notebook.

This notebook goes through all the texts and aplies on them a NLP pipeline consisting of (1) cleaning of the raw text, (2) sentence tokenization, (3) part-of-speech annotation, (4) lemmatization, and (5) named entity recognition.

The processed textual data are saved for future reuse.

In [3]:
import spacy
import os
import glob
from spacy.tokens import Doc
from spacy.language import Language
import pickle
from unidecode import unidecode
import sddk
import pandas as pd
import re
import sys
import importlib
import json
from spacy.tokens import Token
from spacy.language import Language

For preprocessing the latin texts, we will use a module located outside of the current repository, specifically at the same level one level up.

The module can be clonned from here: https://github.com/CCS-ZCU/latin-preprocessing and imported to python following the steps below:

In [4]:
# for preprocessing the latin texts, we will use a module located outside of the current repository, specifically at the same level as the current project.
current_working_directory = os.getcwd()
relative_path = '../../latin-preprocessing/' # change according to your location...
module_path = os.path.abspath(os.path.join(current_working_directory, relative_path))
if module_path not in sys.path:
    sys.path.insert(0, module_path)
# Now import the module
import tomela

CuPy is able to use the GPU.
GPU is available for SpaCy.


Tomela contains tuned latin preprocessing pipeline relying on spaCy and latinCy. You can check the pipeline as here:

In [5]:
tomela.nlp.pipeline

[('senter', <spacy.pipeline.senter.SentenceRecognizer at 0x702cc483af30>),
 ('normer', <function la_core_web_lg.functions.normer(doc)>),
 ('tok2vec', <spacy.pipeline.tok2vec.Tok2Vec at 0x702cc483ac30>),
 ('tagger', <spacy.pipeline.tagger.Tagger at 0x702cc483ab10>),
 ('morphologizer',
  <spacy.pipeline.morphologizer.Morphologizer at 0x702cc483bd70>),
 ('trainable_lemmatizer',
  <spacy.pipeline.edit_tree_lemmatizer.EditTreeLemmatizer at 0x702d6ff3fdd0>),
 ('parser', <spacy.pipeline.dep_parser.DependencyParser at 0x702d6ffcdb60>),
 ('lookup_lemmatizer',
  <function la_core_web_lg.functions.make_lookup_lemmatizer_function(doc)>),
 ('ner', <spacy.pipeline.ner.EntityRecognizer at 0x702cc4c58b30>)]

In [6]:
doc = tomela.nlp("Veritas, vt vlla dicit, semper est universalis et a principiis fundamentalis oritur (lib. 3, cap. VI)")
for token in doc:
    print((token.text, token.lemma_, token.pos_))

('Veritas', 'ueritas', 'NOUN')
(',', ',', 'PUNCT')
('vt', 'vt', 'ADV')
('vlla', 'vllus', 'NOUN')
('dicit', 'dico', 'VERB')
(',', ',', 'PUNCT')
('semper', 'semper', 'ADV')
('est', 'sum', 'AUX')
('universalis', 'uniuersalis', 'ADJ')
('et', 'et', 'CCONJ')
('a', 'ab', 'ADP')
('principiis', 'principium', 'NOUN')
('fundamentalis', 'fundamentalis', 'ADJ')
('oritur', 'orior', 'VERB')
('(', '(', 'PUNCT')
('lib', 'liber', 'NOUN')
('.', '.', 'PUNCT')
('3', '3', 'NUM')
(',', ',', 'PUNCT')
('cap', 'capitulum', 'NOUN')
('.', '.', 'PUNCT')
('VI', 'uis', 'NUM')
(')', ')', 'PUNCT')


In [7]:
source_path = "/srv/data/tome/tome-corpus/emlap_annotated_textblocks/"
len(os.listdir(source_path))

118

In [8]:
os.listdir(source_path)

['Moffett_De_iure_et_praestantia_MDZ_MBS.json',
 'Toxites1567_Spongia_stibii_MDZ_MBS.json',
 'Khunrath1599_Confessio_de_chao_chymicorum_MDZ_MBS_params.json',
 'Anon1550_De_alchemia_opuscula_MDZ_MBS_params.json',
 'Pseudo-Lull1567_Mercuriorum_liber_MDZ_MBS.json',
 'Pantheus1518_Ars_Transmutationis_Metallicae_BL_GB.json',
 'Rossi1585_De_destillatione_MDZ_MBS_params.json',
 'Pseudo-Aquinas1579_Secreta_alchemiae_magnalia_ONB.json',
 'Pseudo-Paracelsus1568_Pyrophilia_vexationumque_ONB.json',
 'Pseudo-Lull1563_Codicillus_MDZ_MBS_pdf_params.json',
 'Bodenstein1559_Isagoge_MDZ_MBS_params.json',
 'Dorn1570_Lapis_metaphysicus_MDZ_MBS_pdf_params.json',
 'Bracesco1548_De_alchemia_dialogi_II_IA_Madrid.json',
 'Pseudo-Lull1518_De_secretis_naturae_MDZ_params.json',
 'Suavius1567_Theophrasti_Paracelsi_Philosophiae_ONB_params.json',
 'Hagecius1585_De_cervisia_ejusque_conficiendi_ratione.json',
 'Pedemontanus1563_De_Secretis_MDZ_MBS.json',
 'Senior1560_De_chemia_senioris_MDZ_MBS_params.json',
 'Phaedro1

In [9]:
files_overview = []
for filename in os.listdir(source_path):
    #filename = 'DuChesne1575_Ad_Iacobi_Auberti_MDZ_Augsburg.json'
    if "_params" not in filename:
        filepath = os.path.join(source_path, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            textblocks = json.load(f)
        pages_n = len(textblocks)
        chars_n = sum([sum([len(tb["text"]) for tb in p]) for p in textblocks])
        files_overview.append({"filename" : filename, "pages_n" : pages_n, "chars_n" : chars_n})
pd.DataFrame(files_overview)

,filename,pages_n,chars_n
0,Moffett_De_iure_et_praestantia_MDZ_MBS.json,115,120620
1,Toxites1567_Spongia_stibii_MDZ_MBS.json,21,14543
2,Pseudo-Lull1567_Mercuriorum_liber_MDZ_MBS.json,405,340776
3,Pantheus1518_Ars_Transmutationis_Metallicae_BL...,53,49495
4,Pseudo-Aquinas1579_Secreta_alchemiae_magnalia_...,69,105063
5,Pseudo-Paracelsus1568_Pyrophilia_vexationumque...,153,136322
6,Bracesco1548_De_alchemia_dialogi_II_IA_Madrid....,131,246242
7,Hagecius1585_De_cervisia_ejusque_conficiendi_r...,65,73530
8,Pedemontanus1563_De_Secretis_MDZ_MBS.json,551,581004
9,Phaedro1562_Aquila_coelestis_MBZ_MBS.json,55,15562



# Develop and test with one example test

In [10]:
filename = 'DuChesne1575_Ad_Iacobi_Auberti_MDZ_Augsburg.json'
filepath = os.path.join(source_path, filename)
with open(filepath, 'r', encoding='utf-8') as f:
    textblocks = json.load(f)

In [11]:
len(textblocks)

93

In [12]:
textblocks[30][:10]

[{'coordinates': [165.1199951171875,
   40.31997299194336,
   427.239990234375,
   48.62395477294922],
  'text': '14\nRESPONSIO\n',
  'tag': 'header'},
 {'coordinates': [165.1199951171875,
   69.11996459960938,
   512.6112670898438,
   73.91996002197266],
  'text': 'calculo aut renum tartaro, tanta vi pro¬\n',
  'tag': 'text'},
 {'coordinates': [165.1199951171875,
   93.3840103149414,
   515.9405517578125,
   98.30400848388672],
  'text': 'desse diceres? Intelligo, Confugeres\n',
  'tag': 'text'},
 {'coordinates': [165.1199951171875,
   116.66397857666016,
   512.2813720703125,
   121.58397674560547],
  'text': 'ad sacram asinorum anchoram, nem¬\n',
  'tag': 'text'},
 {'coordinates': [168.72000122070312,
   140.87997436523438,
   509.6862487792969,
   145.6799774169922],
  'text': 'pe proprietatum occultarum: quod ta¬\n',
  'tag': 'text'},
 {'coordinates': [169.1999969482422,
   165.11996459960938,
   515.7952880859375,
   169.9199676513672],
  'text': 'men ipso sale fieri, qui illos r

In [18]:

# Modify the token extensions for simpler output
if not Token.has_extension("pages"):
    Token.set_extension("pages", default=None)
if not Token.has_extension("textblocks"):
    Token.set_extension("textblocks", default=None)
if not Doc.has_extension("char_to_source"):
    Doc.set_extension("char_to_source", default=None)


def process_textblocks(textblocks):
    full_text = ""
    char_to_source = {}

    for page_idx, page in enumerate(textblocks):
        for tb_idx, tb in enumerate(page):
            if tb["tag"] == "text":
                start_idx = len(full_text)
                text = tomela.text_cleaner(tb["text"])

                for char_idx in range(len(text)):
                    char_to_source[start_idx + char_idx] = {
                        "page_idx": page_idx,
                        "textblock_idx": tb_idx
                    }

                full_text += text

    return full_text, char_to_source


@Language.component("source_tracker")
def source_tracker(doc):
    if doc._.char_to_source is not None:
        for token in doc:
            # Get the character span of the entire token
            token_char_range = range(token.idx, token.idx + len(token.text))

            pages = set()
            textblocks = set()

            for char_idx in token_char_range:
                if char_idx in doc._.char_to_source:
                    source_info = doc._.char_to_source[char_idx]
                    pages.add(source_info["page_idx"])
                    textblocks.add(source_info["textblock_idx"])

            token._.pages = sorted(list(pages))
            token._.textblocks = sorted(list(textblocks))
    return doc


# Add the custom component to your existing pipeline if not already added
if "source_tracker" not in tomela.nlp.pipe_names:
    tomela.nlp.add_pipe("source_tracker", before="senter")


def process_with_source_tracking(textblocks, nlp):
    full_text, char_to_source = process_textblocks(textblocks)
    # Create the doc with the text
    doc = nlp.make_doc(full_text)
    # Set the char_to_source before running the pipeline
    doc._.char_to_source = char_to_source
    # Process the doc through each pipeline component
    for name, proc in nlp.pipeline:
        doc = proc(doc)
    return doc

In [19]:
textblocks[21:22]

[[{'coordinates': [146.63999938964844,
    46.55996322631836,
    435.6400146484375,
    51.599952697753906],
   'text': '5\nAD AVBERTVM.\n',
   'tag': 'header'},
  {'coordinates': [88.55999755859375,
    74.66397857666016,
    435.73681640625,
    79.58397674560547],
   'text': 'naturae iuuandum robur, & aduersus af¬\n',
   'tag': 'text'},
  {'coordinates': [91.19999694824219,
    98.66397857666016,
    444.1780700683594,
    103.58397674560547],
   'text': 'fectus melancholicos, ad exolutum ven¬\n',
   'tag': 'text'},
  {'coordinates': [88.80000305175781,
    122.87997436523438,
    438.64984130859375,
    127.67996978759766],
   'text': 'triculum; ad cardiacos, & praeter ratio¬\n',
   'tag': 'text'},
  {'coordinates': [90.72000122070312,
    146.87997436523438,
    441.6309814453125,
    151.6799774169922],
   'text': 'nem moestos efficax remedium. Certe\n',
   'tag': 'text'},
  {'coordinates': [89.27999877929688,
    172.31997680664062,
    441.3201904296875,
    177.11997985839844

In [20]:
doc = process_with_source_tracking(textblocks[21:22], tomela.nlp)


In [21]:
doc

naturae iuuandum robur, & aduersus affectus melancholicos, ad exolutum uentriculum; ad cardiacos, & praeter rationem moestos efficax remedium. Certein ipsius essentia, quam in tuo auro foliato, multo maiorem facultatem inessemerito credideris. Dabis & illud, miAuberte, in eo purissimo, uim illam occultarum proprietatum maiorem esse,quam in tuis iusculis cum auro coctis.Nec tamen, puto, credes (hoc enim nimis esset absurdum) aurum, quod ne ignis quidem ardore torreri absumiuepotest"(uni enim (ut scribit Poeta) nil deperit auroIgne, uelut solum consumit nulla uetustas,Ac neque rubigo, aut aerugo conficit ulla:Cuncta adeo firmis illic compagibus haerent)"a natiuo calore decoqui aut deuinciita posse, quin cor, integra remanenteillius substantia, ipso corroborari quodammodo queat: quum sit haec Philosophorum sententia, Terram uidelicetomnem esse mortuam, & spiritus rerumin corporibus solos agere posse.Caeterum Laudanum ipsum quanuisopiaticum, non ita tamen conuitiis est

In [22]:
# Test the output
for token in doc[:20]:
    print(f"Token: {token.text}")
    print(f"Page: {token._.pages}")
    print(f"Textblock: {token._.textblocks}")
    print("---")

Token: naturae
Page: [0]
Textblock: [1]
---
Token: iuuandum
Page: [0]
Textblock: [1]
---
Token: robur
Page: [0]
Textblock: [1]
---
Token: ,
Page: [0]
Textblock: [1]
---
Token: &
Page: [0]
Textblock: [1]
---
Token: aduersus
Page: [0]
Textblock: [1]
---
Token: affectus
Page: [0]
Textblock: [1, 2]
---
Token: melancholicos
Page: [0]
Textblock: [2]
---
Token: ,
Page: [0]
Textblock: [2]
---
Token: ad
Page: [0]
Textblock: [2]
---
Token: exolutum
Page: [0]
Textblock: [2]
---
Token: uentriculum
Page: [0]
Textblock: [2, 3]
---
Token: ;
Page: [0]
Textblock: [3]
---
Token: ad
Page: [0]
Textblock: [3]
---
Token: cardiacos
Page: [0]
Textblock: [3]
---
Token: ,
Page: [0]
Textblock: [3]
---
Token: &
Page: [0]
Textblock: [3]
---
Token: praeter
Page: [0]
Textblock: [3]
---
Token: rationem
Page: [0]
Textblock: [3, 4]
---
Token: moestos
Page: [0]
Textblock: [4]
---


In [20]:
tomela.nlp.pipeline

[('source_tracker', <function __main__.source_tracker(doc)>),
 ('senter', <spacy.pipeline.senter.SentenceRecognizer at 0x7a105781e3f0>),
 ('normer', <function la_core_web_lg.functions.normer(doc)>),
 ('tok2vec', <spacy.pipeline.tok2vec.Tok2Vec at 0x7a105781dfd0>),
 ('tagger', <spacy.pipeline.tagger.Tagger at 0x7a105781e450>),
 ('morphologizer',
  <spacy.pipeline.morphologizer.Morphologizer at 0x7a105781ed50>),
 ('trainable_lemmatizer',
  <spacy.pipeline.edit_tree_lemmatizer.EditTreeLemmatizer at 0x7a105781ee70>),
 ('parser', <spacy.pipeline.dep_parser.DependencyParser at 0x7a10be1c86d0>),
 ('lookup_lemmatizer',
  <function la_core_web_lg.functions.make_lookup_lemmatizer_function(doc)>),
 ('ner', <spacy.pipeline.ner.EntityRecognizer at 0x7a1057456570>)]

In [17]:
def process_textblocks(textblocks):
    # Initialize empty text and mapping dictionaries
    full_text = ""
    char_to_source = {}

    # Process each page
    for page_idx, page in enumerate(textblocks):
        # Process each textblock in the page
        for tb_idx, tb in enumerate(page):
            if tb["tag"] == "text":
                # For each character in the textblock, store its source
                start_idx = len(full_text)
                text = tb["text"]

                for char_idx in range(len(text)):
                    abs_idx = start_idx + char_idx
                    char_to_source[abs_idx] = {
                        "page_idx": page_idx,
                        "textblock_idx": tb_idx
                    }

                full_text += text

    return full_text, char_to_source


In [21]:
test_output = process_textblocks(textblocks)

('AVBERTI VINDO¬\nNIS DE ORTV ET CAVSIS\nMETALLORVM CONTRA\nChymicos Explicationem\nIOSEPHI QVERCETANI ARME¬\nniaci, D. Medici breuis Responsio.\nEIVSDEM DE EXQVISITA\nMineralium, Animalium, & Vegetabilium me¬\ndicamentorum Spagyrica praeparatione &\nvsu, perspicua Tractatio.\nLVGDVNI,\nApud loannem Lertotium.\nM. D. LXXV.\n[F] En nostre estat au vostre tout contraire,\nSi nous soufflons, vous humez d\'autrepart:\nOr sus enfans, de ces deux poincts de l\'art,\nIugez lequel est plus seant de faire. [/F]\nVIRTVTIS COMES\nINVIDIA.\nSIMO AC SPLENDI¬\ndissimo viro Jacobo de la\nFin, Regii ordinis Equiti\nAurato, eiusque Nobili\ncubiculario, D. de la Fin la¬\nNocle, Pluuiers, Baroni\nd Aubusson, &c. Josephus\nQuercetanus. S.\nMemoriae proditum est,\nPythagoram, hominum\nvitam dixisse consimi¬\nlem sibi videri eius Pa¬\nnegyris ac mercatus Graeciae no¬\nbilissimi, quò nonnulli certandi,\nalij emendi & vendendi, alij ve¬\nrò spectandi tantùm causa se con¬\nferebant: Philosophos autem eos esse\

In [ ]:

# lets encapsulate the cleaning and spacy pipeline application into one function
def from_textblocks_to_doc(cleaned_textblocks_pages, lowertext=False):
    #cleantext = text_cleaner(rawtext, lowertext)
    segment_len = 800000
    if len(cleaned_textblocks_pages) > 400:

    if len(cleantext) > segment_len:
        segment_docs = []
        parts = cleantext[:segment_len].rpartition(". ")
        current_segment = parts[0] + parts[1]
        segment_doc = nlp(current_segment)
        segment_docs.append(segment_doc)
        next_segment_beginning = parts[2]
        for n in range(segment_len, len(cleantext), segment_len):
            segment = cleantext[n:n+segment_len]
            if len(segment) == segment_len:
                parts = cleantext[n:n+segment_len].rpartition(". ")
                current_segment = parts[0] + parts[1]
                segment_doc = nlp(next_segment_beginning + current_segment)
                next_segment_beginning = parts[2]
            else:
                segment_doc = nlp(segment)
            segment_docs.append(segment_doc)
        doc = Doc.from_docs(segment_docs)
    else:
        doc = nlp(cleantext)
    return doc




In [ ]:
def create_source_aware_nlp(textblocks, base_nlp):
    # Process textblocks and get character mapping
    full_text, char_to_source = process_textblocks(textblocks)

    # Create a closure over the character mapping
    @Language.component("source_tracker")
    def source_tracker(doc):
        for token in doc:
            # Get the character index at the start of the token
            char_idx = token.idx

            # Look up the source information
            if char_idx in char_to_source:
                source_info = char_to_source[char_idx]
                token._.page_idx = source_info["page_idx"]
                token._.textblock_idx = source_info["textblock_idx"]

        return doc

    # Create a new pipeline with the source tracker
    nlp = base_nlp.from_config()
    nlp.add_pipe("source_tracker", before="senter")

    return nlp, full_text

In [43]:
test loading one specific text
fn = fns[0][1]
with open(fn, "r", encoding="utf-8") as f:
    rawtext = f.read()
rawtext[:1000]

"MARIA\n\n\nAVLA PVRIFICATIONIS AVRI\nNON SINE SALE & NON SINE ARGILLA\nVOARCH ¬\nADVMIA\ncontra Alchi'miam : Ars distincta ab\nArchimi'a, & Sophia: cum Additio¬\nnibus & Proportionibus: Numeris: &\nFiguris oportubit Ioannis Augustini\nPanthei Veneti sacerdotis.\nVenetiis. Diebus. Aprilis.\nM. D. XXX.\n\n\nMORIENVS\n\n\nCONCESSIO IMPRESSIONIS.\nConcessio Reuerendissimi. D. Legati apostolici.\nALTOBELLVS AVEROL¬\nDVS Dei & apostolicae sedis gratia\nEpiscopus Polen. S. D. N. Papae Re¬\nferen. & per totum Venetorum domi¬\nnium, cum potestate Legati Cardina¬\nIlis de latere, Legatus apostolicus. Dile¬\ncto nobis in CHRISTO Ioanni Augustino Pantheo\nVeneto sacerdoti, salutem in domino sempiternam: & caetera.\nMandamus igitur & praecipimus authoritate apostolica,\nqua ex munere legationis nostrae huiusmodi fungimur in\nhac parte: ne quis legationi nostrae subiectus: id ipsum\nopusculum siue Latina, siue Vernacula lingua perscriptum:\nin locis legationis nostrae huiusmodi imprimere, aut im¬\n

In [14]:
# in case you locally tune the module, ensure that you have loaded the latest version!
importlib.reload(tomela)

CuPy is able to use the GPU.
GPU is available for SpaCy.


<module 'tomela' from '/home/jupyter-vojta/notebooks/latin-preprocessing/tomela/__init__.py'>

In [15]:
tomela.text_cleaner(rawtext)[:5000]

"Maria Aula Purificationis Auri Non Sine Sale & Non Sine Argilla Uoarch Adumia contra Alchi'miam : Ars distincta ab Archimi'a, & Sophia: cum Additionibus & Proportionibus: Numeris: & Figuris oportubit Ioannis Augustini Panthei Ueneti sacerdotis. Uenetiis. Diebus. Aprilis. M. D. Xxx. Morienus Concessio Impressionis. Concessio Reuerendissimi. D. Legati apostolici. Altobellus Aueroldus Dei & apostolicae sedis gratia Episcopus Polen. S. D. N. Papae Referen, & per totum Uenetorum dominium, cum potestate Legati Cardinailis de latere, Legatus apostolicus. Dilecto nobis in Christo Ioanni Augustino Pantheo Ueneto sacerdoti, salutem in domino sempiternam: & caetera. Mandamus igitur & praecipimus authoritate apostolica, qua ex munere legationis nostrae huiusmodi fungimur in hac parte: ne quis legationi nostrae subiectus: id ipsum opusculum siue Latina, siue Uernacula lingua perscriptum: in locis legationis nostrae huiusmodi imprimere, aut impressum uenundare, uendendumue tradere ullis in locis al

In [16]:
doc = tomela.from_rawtext_to_doc(rawtext, lowertext=False)

In [17]:
target_path = "/srv/data/tome/tome-corpus/sents_data_jsons_v2-0/"
try:
    os.mkdir(target_path)
except:
    pass

In [44]:
%%time
for id, fn in fns:
    with open(fn, "r", encoding="utf-8") as f:
        rawtext = f.read()
    doc = tomela.from_rawtext_to_doc(rawtext)
    doc_sentdata = [(sent.text, [(t.text, t.lemma_, t.pos_, (t.idx - sent[0].idx, t.idx - sent[0].idx + len(t))) for t in sent]) for sent in doc.sents]
    sent_data_updated = []
    for n_sent, sent_data in enumerate(doc_sentdata):
        sent_data_updated.append((id, n_sent, sent_data[0], sent_data[1]))
    with open(target_path + id + ".json", "w") as f:
        json.dump(sent_data_updated, f)

CPU times: user 2min 47s, sys: 7.81 s, total: 2min 55s
Wall time: 2min 54s


In [34]:
# Extract the lemmatized sentences

In [45]:
fns_jsons = os.listdir(target_path)
fns_jsons[:10]

['100044.json',
 '100034.json',
 '100014.json',
 '100060.json',
 '100010.json',
 '100043.json',
 '100041.json',
 '100012.json',
 '100025.json',
 '100019.json']

In [46]:
sents_data = json.load(open(target_path + fns_jsons[4], "r"))
sents_data[100:103]

[['100010',
  100,
  'Et eodem libro, cap. 3, scripsimus, eum, qui non habuerit ingenium naturale, & animum, ingeniose & subtiliter perscrutantem principia naturalia, & naturae fundamenta, & artificia, quae naturam assequi possint, in suae actionis proprietatibus, non inuenturum huius preciosissimae artis & magisterii ueram radicem.',
  [['Et', 'et', 'CCONJ', [0, 2]],
   ['eodem', 'idem', 'DET', [3, 8]],
   ['libro', 'liber', 'NOUN', [9, 14]],
   [',', ',', 'PUNCT', [14, 15]],
   ['cap', 'capitulum', 'NOUN', [16, 19]],
   ['.', '.', 'PUNCT', [19, 20]],
   ['3', '3', 'NUM', [21, 22]],
   [',', ',', 'PUNCT', [22, 23]],
   ['scripsimus', 'scribo', 'VERB', [24, 34]],
   [',', ',', 'PUNCT', [34, 35]],
   ['eum', 'is', 'PRON', [36, 39]],
   [',', ',', 'PUNCT', [39, 40]],
   ['qui', 'qui', 'PRON', [41, 44]],
   ['non', 'non', 'PART', [45, 48]],
   ['habuerit', 'habeo', 'VERB', [49, 57]],
   ['ingenium', 'ingenium', 'NOUN', [58, 66]],
   ['naturale', 'naturalis', 'ADJ', [67, 75]],
   [',', ','

In [47]:
lemmatized_sents_path = "/srv/data/tome/tome-corpus/lemmatized_sents_v2-0/"
try:
    os.mkdir(lemmatized_sents_path)
except:
    pass

In [49]:

for fn in fns_jsons:
    lemmatized_sents = []
    sents_data = json.load(open(target_path + fn, "rb"))
    for (doc_id, sent_id, sent_text, sent_data) in sents_data:
        lemmasent = []
        for wordform, lemma, tag, position in sent_data:
            if tag in ["NOUN", "PROPN", "ADJ", "VERB"]:
                lemmasent.append(lemma)
        lemmatized_sents.append(" ".join(lemmasent) + "\n")
    with open(lemmatized_sents_path + fn.replace(".json", ".txt"), "w", encoding="utf-8") as f:
        f.writelines(lemmatized_sents)